<table style="width:100%"> <tr> <td style="vertical-align:middle; text-align:left;"> <font size="2"> Дополнительный код для книги <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> (Создание большой языковой модели с нуля) от <a href="https://sebastianraschka.com">Себастьяна Рашки</a><br> <br>Репозиторий с кодом: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a> </font> </td> <td style="vertical-align:middle; text-align:left;"> <a href="https://sebastianraschka.com"> <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"> </a> </td> </tr> </table>

# Глава 2. Работа с текстовыми данными

In [3]:
import sys
print("Путь к Python:", sys.executable)
print("Версия Python:", sys.version)

Путь к Python: c:\Users\seera\AppData\Local\Python\pythoncore-3.14-64\python.exe
Версия Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]


In [4]:
import sys
import subprocess

# Проверяем установленные пакеты
result = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True)
print(result.stdout)

Package                 Version
----------------------- -----------
asttokens               3.0.1
certifi                 2026.4.22
charset-normalizer      3.4.7
colorama                0.4.6
comm                    0.2.3
debugpy                 1.8.20
decorator               5.2.1
executing               2.2.1
filelock                3.29.0
fsspec                  2026.3.0
idna                    3.13
ipykernel               7.2.0
ipython                 9.13.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
Jinja2                  3.1.6
jupyter_client          8.8.0
jupyter_core            5.9.1
MarkupSafe              3.0.3
matplotlib-inline       0.2.1
mpmath                  1.3.0
nest-asyncio            1.6.0
networkx                3.6.1
packaging               26.2
parso                   0.8.6
pip                     25.3
platformdirs            4.9.6
prompt_toolkit          3.0.52
psutil                  7.2.2
pure_eval               0.2.3
Pygments               

In [5]:
import sys
import subprocess

# Установка torch и tiktoken
packages = ["torch", "tiktoken"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [6]:
from importlib.metadata import version

print("Версия torch:", version("torch"))
print("Версия tiktoken:", version("tiktoken"))

Версия torch: 2.11.0
Версия tiktoken: 0.12.0


- Задача: подготовить данные и выборку образцов, чтобы "подготовить" входные данные для LLM

<img src="https://camo.githubusercontent.com/904aadb708a5c5d3ab94e96f2a89df661b677aee26f5292c8050261da68829ea/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30312e776562703f74696d657374616d703d31" width="800px">

## 2.1 Векторное представление слов

Существует множество форм вложения (эмбеддинг). Текущая задача - вложение текста

<img src="https://camo.githubusercontent.com/3e71d5547925bc40be3ad13ddebab7b8b39507f75af0a19db521d1e2e054fc4d/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30322e77656270" width="800px">

- LLM работают с вложениями в многомерные пространства (т.е. в тысячи измерений)
- Человек не может визуализировать такие многомерные пространства (люди мыслят в 1, 2 или 3 измерениях). На рисунке ниже показано 2-мерное пространство для вложений

<img src="https://camo.githubusercontent.com/2de90c75263eadcfe63a51e77da3f8cc86a3bbb98507e24de725629f82f20197/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30332e77656270" width="800px">

## 2.2 Токенизация текста

- Задача: разбить текст на более мелкие части, такие как отдельные слова и знаки препинания

<img src="https://camo.githubusercontent.com/937931bc39d9211a7d16803fcea8ce621422a635f72934af703468774e30bcd7/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30342e77656270" width="800px">

- Задача: загрузить необработанный текст для работы
- ["Вердикт" Эдит Уортон](https://en.wikisource.org/wiki/The_Verdict) - это рассказ, находящийся в открытом доступе

In [ ]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    # Отправка GET-запрос по указанному URL с таймаутом 30 секунд
    # Если сервер не ответит за это время, запрос прервётся с ошибкой 
    response = requests.get(url, timeout=30) 

    # Проверка статус ответа
    # Если сервер вернул ошибку (4xx или 5xx), сразу выбрасывается исключение
    # Это быстрый способ проверить, что запрос успешен (2xx), без ручной проверки кода
    response.raise_for_status()

    # Открыть (или создаём) файл по указанному пути `file_path` для записи в бинарном режиме ("wb" — write binary)
    # Менеджер контекста `with` гарантирует автоматическое закрытие файла, даже если при записи возникнет ошибка
    # Файловый объект доступен через переменную `f`.
    with open(file_path, "wb") as f:
        f.write(response.content)

In [8]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Общее количество символов:", len(raw_text))
print(raw_text[:99])

Общее количество символов: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- Цель состоит в том, чтобы разметить и внедрить этот текст для LLM
- Задача: разработать простой разметчик на основе простого примера текста, который можно позже применить к тексту выше
- Следующее регулярное выражение будет разделяться пробелами

In [9]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- Задача: разделить текст не только на пробелы, но и на запятые и точки

In [10]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- Задача: удалить пустые строки

In [11]:
# Удалить пробелы из каждого элемента, а затем отфильтровать все пустые строки
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- Задача: разобраться с другими типами знаков препинания, такими как точки, вопросительные знаки и так далее

In [12]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- Задача: применить текущую токенизацию к исходному тексту

<img src="https://camo.githubusercontent.com/322bfa4d8bce2fd2116543d2e8c8cb3732b87a0a676c55f8f959b33ab2c6c7c9/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30352e77656270" width="800px">

In [13]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- Задача: рассчитать общее количество токенов

In [14]:
print(len(preprocessed))

4690


## 2.3 Преобразование токенов в идентификаторы токенов

- Задача: пребразовать текстовые токены в идентификаторы токенов, которые позже можно обработать с помощью слоев встраивания

<img src="https://camo.githubusercontent.com/cbf2eaec29afc698a1fbe9839967d5fa07dfd07dc35482fa5298de03fc33af4b/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30362e77656270" width="800px">

- Задача: составить словарь из всех текущих уникальных токенов

In [15]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [16]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- Ниже приведены первые 50 значений из словаря:

In [17]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- Ниже иллюстрация разбиения на символы короткого образца текста с использованием небольшого словарного запаса:

<img src="https://camo.githubusercontent.com/2f430af2ba9dce8e6b3fe730ec97235878c689db0c57003e5cb30eb1ab2778d8/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30372e776562703f313233" width="800px">

- Задача: создать класс-токенизатор и добавить туда текущий словарь

In [18]:
class SimpleTokenizerV1:

    # Конструктор класса
    # Вызывается при создании объекта
    # self - ссылка на сам объект
    # Принимает словарь vocab, где ключ — слово, значение — его ID.
    def __init__(self, vocab):

        # Сохранение словаря для прямого преобразования: строка → число
        self.str_to_int = vocab

        # Создание обратного словаря (число → строка) с помощью генератора: для каждой пары s,i в vocab.items() создать запись {i: s}
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        # Замена пробелов перед указанными знаками препинания
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

- Функция "encode" преобразует текст в идентификаторы токенов
- Функция "decode" преобразует идентификаторы токенов обратно в текст

<img src="https://camo.githubusercontent.com/343b55a9079b960142914b9b1c795535668caeb8facd8fe48bd8c36bbd7162f3/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30382e776562703f313233" width="800px">

- Можно использовать токенизатор для кодирования (то есть токенизации) текстов в целые числа
- Эти целые числа затем могут быть встроены (позже) в качестве входных данных для LLM

In [19]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- Можно декодировать целые числа обратно в текст

In [20]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [21]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'